# 🐞 LadyBug — ComfyUI na nuvem (Google Colab GPU)

Roda o ComfyUI numa GPU boa (T4 16GB grátis, ou L4/A100 no Pro) e abre no seu Chrome por um link.
É a mesma coisa da sua 3050, mas com placa de verdade — os modelos bons cabem e o vídeo fica bonito.

**Como usar:** Ambiente de execução → Alterar tipo → GPU. Depois roda as células em ordem (▶). No fim aparece um link `https://...trycloudflare.com` — abre no Chrome e é o ComfyUI.

In [ ]:
#@title 1) Confere a GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
#@title 2) Instala ComfyUI + custom nodes (AnimateDiff, IPAdapter, VideoHelper)
%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI 2>/dev/null || echo ok
%cd /content/ComfyUI
!pip -q install -r requirements.txt
%cd custom_nodes
!git clone https://github.com/ltdrdata/ComfyUI-Manager 2>/dev/null || echo ok
!git clone https://github.com/Kosinkadink/ComfyUI-AnimateDiff-Evolved 2>/dev/null || echo ok
!git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite 2>/dev/null || echo ok
!git clone https://github.com/cubiq/ComfyUI_IPAdapter_plus 2>/dev/null || echo ok
!pip -q install opencv-python-headless imageio-ffmpeg
%cd /content/ComfyUI

In [ ]:
#@title 3) Baixa os modelos (checkpoint de ARTE + movimento + consistência)
import os
def baixa(url, dest):
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    if not os.path.exists(dest): os.system(f'wget -q -O "{dest}" "{url}"')
    print(('OK  ' if os.path.exists(dest) and os.path.getsize(dest)>1e6 else 'FALHOU '), dest)
# DreamShaper 8 — SD1.5 de arte, segura personagem (bem melhor que o base cru)
baixa('https://huggingface.co/Lykon/DreamShaper/resolve/main/DreamShaper_8_pruned.safetensors', 'models/checkpoints/dreamshaper_8.safetensors')
# AnimateDiff v3 (movimento)
baixa('https://huggingface.co/guoyww/animatediff/resolve/main/v3_sd15_mm.ckpt', 'models/animatediff_models/v3_sd15_mm.ckpt')
# IPAdapter (consistencia de personagem) + CLIP vision
baixa('https://huggingface.co/h94/IP-Adapter/resolve/main/models/ip-adapter_sd15.safetensors', 'models/ipadapter/ip-adapter_sd15.safetensors')
baixa('https://huggingface.co/h94/IP-Adapter/resolve/main/models/image_encoder/model.safetensors', 'models/clip_vision/CLIP-ViT-H-14-laion2B.safetensors')

In [ ]:
#@title 4) Sobe o ComfyUI e cria o link pro Chrome (cloudflared)
!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared
import subprocess, threading, time, re
srv = subprocess.Popen(['python','main.py','--listen','127.0.0.1','--port','8188'], cwd='/content/ComfyUI')
time.sleep(25)
tun = subprocess.Popen(['/content/cloudflared','tunnel','--url','http://127.0.0.1:8188','--no-autoupdate'],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print('Abrindo o tunel... (link em ~10s)')
for _ in range(60):
    line = tun.stdout.readline()
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if m:
        print('\n=========================================')
        print('  ABRA ISTO NO SEU CHROME:', m.group(0))
        print('=========================================')
        break

## 5) Rodar os workflows da LadyBug
No ComfyUI que abriu no link: menu **Workflow → Open** e escolha os `.json` da pasta `ladybug-workflows`
(arraste eles pro Colab em `Arquivos` na lateral, ou cole o conteúdo). Ordem:

- `01-imagem.json` — imagem base
- `02-personagem.json` — personagem consistente (IPAdapter)
- `03-video.json` — imagem→vídeo (AnimateDiff)
- `05-personagem-video.json` — a **peça-mãe** (personagem consistente + movimento)

Troca o checkpoint pra `dreamshaper_8.safetensors` nos nós (fica muito melhor que na 3050).
Deixa a aba do link aberta — você vê a geração acontecendo ao vivo.